In [1]:
%%writefile app.py
#%pip install ucimlrepo
#%pip install scikit-learn

from ucimlrepo import fetch_ucirepo 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error 
import seaborn as sns

online_retail = fetch_ucirepo(id=352) 

df = pd.DataFrame(online_retail.data.features)

print("Rows:",df.shape[0]) 
print("Total Features:", df.shape[1])
print("Columns:")
df.head()

Overwriting app.py


In [ ]:
print("Unique Countries:")
#print(df['Country'].unique())

Unique Countries:


NameError: name 'df' is not defined

In [ ]:
df[df['Quantity'] < 0]['Quantity'].count()

np.int64(10624)

In [ ]:
df[df['UnitPrice'] < 0]['UnitPrice'].count()

np.int64(2)

In [ ]:
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df_clean[df_clean['Quantity'] < 0]['Quantity'].count()

np.int64(0)

In [ ]:
df_clean[df_clean['UnitPrice'] < 0]['UnitPrice'].count()

np.int64(0)

In [ ]:
data_clean = df_clean[['Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'Country']]

In [ ]:
data_clean.head()

,Description,Quantity,InvoiceDate,UnitPrice,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,United Kingdom


In [ ]:
data_clean.isnull().sum()

Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

In [ ]:
print("🔍 AUDITING FOR HIDDEN PLACEHOLDERS...\n")

# 1. Search for hidden text placeholders in ALL columns at once
text_placeholders = ['unspecified', 'unknown', 'none', 'null', 'missing']

for placeholder in text_placeholders:
    # Check if the placeholder text exists anywhere in the dataframe (case-insensitive)
    matches = data_clean.apply(lambda col: col.astype(str).str.lower().str.contains(placeholder)).sum().sum()
    if matches > 0:
        print(f"⚠️ Found {matches} instances of the text '{placeholder}' hidden inside your columns!")
    else:
        print(f"✅ Clean: No instances of '{placeholder}' found.")

print("\n🔢 AUDITING FOR FAKE NUMERICAL ID NUMBERS...")

# 2. Check if there are any suspicious fake numeric IDs or entries
# Since Quantity and UnitPrice must be > 0, let's verify no extreme placeholder outliers exist
zero_count = (data_clean[['Quantity', 'UnitPrice']] == 0).sum().sum()
print(f"✅ Clean: Found {zero_count} exact zeros in Quantity or UnitPrice.")

max_qty = data_clean['Quantity'].max()
max_price = data_clean['UnitPrice'].max()
print(f"📊 Maximum structural values found: Max Quantity = {max_qty:,} | Max Unit Price = ${max_price:,.2f}")

🔍 AUDITING FOR HIDDEN PLACEHOLDERS...

⚠️ Found 446 instances of the text 'unspecified' hidden inside your columns!
✅ Clean: No instances of 'unknown' found.
✅ Clean: No instances of 'none' found.
✅ Clean: No instances of 'null' found.
✅ Clean: No instances of 'missing' found.

🔢 AUDITING FOR FAKE NUMERICAL ID NUMBERS...
✅ Clean: Found 0 exact zeros in Quantity or UnitPrice.
📊 Maximum structural values found: Max Quantity = 80,995 | Max Unit Price = $13,541.33


In [ ]:
data_clean[data_clean['Description'].astype(str).str.lower() == 'unspecified']['Description'].count()

np.int64(0)

In [ ]:
data_clean[data_clean['Quantity'].astype(str).str.lower()  == 'unspecified']['Quantity'].count()

np.int64(0)

In [ ]:
data_clean[data_clean['InvoiceDate'].astype(str).str.lower()  == 'unspecified']['InvoiceDate'].count()

np.int64(0)

In [ ]:
data_clean[data_clean['UnitPrice'].astype(str).str.lower()  == 'unspecified']['UnitPrice'].count()

np.int64(0)

In [ ]:
data_clean[data_clean['Country'].astype(str).str.lower()  == 'unspecified']['Country'].count()

np.int64(446)

In [ ]:
data_final = data_clean[data_clean['Country'].astype(str).str.lower()  != 'unspecified']
data_final[data_final['Country'].astype(str).str.lower()  == 'unspecified']['Country'].count()

np.int64(0)

In [ ]:
#data_final.groupby('Country')['Quantity'*'UnitPrice'].sum().sort_values(ascending=False).head(10)

In [ ]:
data_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 529658 entries, 0 to 541908
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Description  529658 non-null  object 
 1   Quantity     529658 non-null  int64  
 2   InvoiceDate  529658 non-null  object 
 3   UnitPrice    529658 non-null  float64
 4   Country      529658 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 24.2+ MB


In [ ]:
data_final.assign(Revenue = data_final['Quantity'] * data_final['UnitPrice']).groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

Country
United Kingdom    9025222.084
Netherlands        285446.340
EIRE               283453.960
Germany            228867.140
France             209715.110
Australia          138521.310
Spain               61577.110
Switzerland         57089.900
Belgium             41196.340
Sweden              38378.330
Name: Revenue, dtype: float64

In [ ]:
data_final['Target_Revenue'] = data_final['Quantity'] * data_final['UnitPrice']

/var/folders/74/87zgzy_911q164s59j3jb5nr0000gn/T/ipykernel_45498/1521744966.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_final['Target_Revenue'] = data_final['Quantity'] * data_final['UnitPrice']


In [ ]:
country_0 = data_final[data_final['Country'].astype(str).str.lower()  == 'united kingdom'].copy()
country_1 = data_final[data_final['Country'].astype(str).str.lower()  == 'netherlands'].copy()
country_2 = data_final[data_final['Country'].astype(str).str.lower()  == 'eire'].copy()

In [ ]:
def train_and_evaluate_region(region_df, region_name):
    # Isolate your features (X) and target (y)
    X = region_df[['Quantity', 'UnitPrice']]
    y = region_df['Target_Revenue']
    
    # 2.1 Split the data into a training set and validation set at a ratio of 75:25
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
    
    # 2.2 Train the model and make predictions for the validation set
    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    
    # 2.3 Save the predictions and correct answers for the validation set
    results = pd.DataFrame({
        'true_values': y_val,
        'predictions': predictions
    })
    
    # 2.4 Calculate the average volume of predicted reserves (revenue) and model RMSE
    avg_predicted_revenue = results['predictions'].mean()
    rmse = np.sqrt(mean_squared_error(results['true_values'], results['predictions']))
    
    # Print metrics exactly as requested in your instructions
    print(f"📊 {region_name} Model Evaluation:")
    print(f"    Average Predicted Transaction Value: ${avg_predicted_revenue:,.2f}")
    print(f"    Model RMSE: ${rmse:,.2f}\n")
    
    return results

# Run the machine learning model loops!
results_r0 = train_and_evaluate_region(country_0, "Region 0 (United Kingdom)")
results_r1 = train_and_evaluate_region(country_1, "Region 1 (Netherlands)")
results_r2 = train_and_evaluate_region(country_2, "Region 2 (EIRE)")

📊 Region 0 (United Kingdom) Model Evaluation:
    Average Predicted Transaction Value: $18.54
    Model RMSE: $57.36

📊 Region 1 (Netherlands) Model Evaluation:
    Average Predicted Transaction Value: $119.87
    Model RMSE: $99.60

📊 Region 2 (EIRE) Model Evaluation:
    Average Predicted Transaction Value: $35.72
    Model RMSE: $45.30



Region 2 has the lowest RMSE with quite a high Average Predicted Trasaction Value. This is our best choice of the three options. Though Region 1 has a high Average Predicted Transaction Value, it has a very high Model RMSE, which is not ideal. EIRE in this case it our top selection for our high-stakes market, and we can plan a physical, high-capital infrastructure expansion with this location.

In [ ]:
"""# Global parameters from Step 3
BUDGET = 500000.00         # The $500,000 cost to open the logistics hub
TARGET_ORDERS = 200        # We pick the top 200 highest-value transactions

# Step 4: Core profit calculation function (OilyGiant Step 4 structure!)
def calculate_profit(target_true, target_predictions, count=TARGET_ORDERS):
    # Create a dataframe to align the AI's guesses with reality
    data = pd.DataFrame({
        'predicted': target_predictions,
        'actual': target_true
    })
    
    # 4.1 Pick the top 200 transactions based on the model's predictions
    top_200 = data.sort_values(by='predicted', ascending=False).head(count)
    
    # 4.2 Sum the ACTUAL real-world transaction revenue of these specific orders
    total_actual_revenue = top_200['actual'].sum()
    
    # 4.3 Calculate net profit (Total revenue minus our capital investment budget)
    # Note: No oil multipliers needed here! The revenue is already calculated in dollars.
    net_profit = total_actual_revenue - BUDGET
    
    return net_profit

# Calculate and print the net profit for your 3 clean retail markets!
profit_region_0 = calculate_profit(results_r0['true_values'], results_r0['predictions'])
print(f"💰 Net Profit for Region 0 (United Kingdom): ${profit_region_0:,.2f}")

profit_region_1 = calculate_profit(results_r1['true_values'], results_r1['predictions'])
print(f"💰 Net Profit for Region 1 (Netherlands):    ${profit_region_1:,.2f}")

profit_region_2 = calculate_profit(results_r2['true_values'], results_r2['predictions'])
print(f"💰 Net Profit for Region 2 (EIRE / Ireland):  ${profit_region_2:,.2f}")"""

'# Global parameters from Step 3\nBUDGET = 500000.00         # The $500,000 cost to open the logistics hub\nTARGET_ORDERS = 200        # We pick the top 200 highest-value transactions\n\n# Step 4: Core profit calculation function (OilyGiant Step 4 structure!)\ndef calculate_profit(target_true, target_predictions, count=TARGET_ORDERS):\n    # Create a dataframe to align the AI\'s guesses with reality\n    data = pd.DataFrame({\n        \'predicted\': target_predictions,\n        \'actual\': target_true\n    })\n    \n    # 4.1 Pick the top 200 transactions based on the model\'s predictions\n    top_200 = data.sort_values(by=\'predicted\', ascending=False).head(count)\n    \n    # 4.2 Sum the ACTUAL real-world transaction revenue of these specific orders\n    total_actual_revenue = top_200[\'actual\'].sum()\n    \n    # 4.3 Calculate net profit (Total revenue minus our capital investment budget)\n    # Note: No oil multipliers needed here! The revenue is already calculated in dollars.\n 

In [ ]:
# Global Financial Constants from Step 3
BUDGET = 5000.00
TARGET_COUNT = 200

# Step 4: Core profit calculation function (OilyGiant Step 4)
def calculate_profit(target_true, target_predictions, count=TARGET_COUNT):
    # Sort predictions from highest to lowest and find the index positions
    preds_sorted = target_predictions.sort_values(ascending=False)
    # Match those top index positions with the actual real-world revenue values
    selected_true = target_true.loc[preds_sorted.index].head(count)
    # Calculate revenue minus the operational investment budget
    return selected_true.sum() - BUDGET

# Step 5: Bootstrapping simulation engine (OilyGiant Step 5)
def bootstrap_region(results_df, region_name):
    state = np.random.RandomState(42)
    profits = []
    
    # Run 1,000 independent campaign simulations
    for i in range(1000):
        # Sample 500 random transactions to represent regional baseline risk
        sample_true = results_df['true_values'].sample(n=500, replace=True, random_state=state)
        sample_preds = results_df['predictions'].loc[sample_true.index]
        
        # Calculate profit for this specific simulated run
        profit = calculate_profit(sample_true, sample_preds)
        profits.append(profit)
        
    profits = pd.Series(profits)
    
    # Calculate final OilyGiant risk parameters
    mean_profit = profits.mean()
    lower_ci = profits.quantile(0.025)
    upper_ci = profits.quantile(0.975)
    
    # Calculate Risk of Loss (percentage of simulations where profit was negative)
    risk_of_loss = (profits < 0).mean() * 100
    
    print(f"📈 {region_name} Bootstrapping Summary:")
    print(f"    Average Projected Net Profit: ${mean_profit:,.2f}")
    print(f"    95% Confidence Interval:     (${lower_ci:,.2f} to ${upper_ci:,.2f})")
    print(f"    Strict Risk of Losses:        {risk_of_loss:.2f}%")
    
    # Flag if the region violates the company's 2.5% maximum loss threshold
    if risk_of_loss > 2.5:
        print("    ❌ REJECTED: Risk of loss exceeds company's 2.5% structural threshold.\n")
    else:
        print("    ✅ APPROVED: Safe risk profile for capital investment.\n")
        
    return profits

print("⏳ Running 1,000 Bootstrap simulations across all regions...\n")
profits_r0 = bootstrap_region(results_r0, "Region 0 (United Kingdom)")
profits_r1 = bootstrap_region(results_r1, "Region 1 (Netherlands)")
profits_r2 = bootstrap_region(results_r2, "Region 2 (EIRE / Ireland)")

⏳ Running 1,000 Bootstrap simulations across all regions...

📈 Region 0 (United Kingdom) Bootstrapping Summary:
    Average Projected Net Profit: $2,417.78
    95% Confidence Interval:     ($301.49 to $7,125.66)
    Strict Risk of Losses:        1.10%
    ✅ APPROVED: Safe risk profile for capital investment.

📈 Region 1 (Netherlands) Bootstrapping Summary:
    Average Projected Net Profit: $43,387.28
    95% Confidence Interval:     ($32,110.90 to $59,704.42)
    Strict Risk of Losses:        0.00%
    ✅ APPROVED: Safe risk profile for capital investment.

📈 Region 2 (EIRE / Ireland) Bootstrapping Summary:
    Average Projected Net Profit: $9,498.56
    95% Confidence Interval:     ($5,649.09 to $15,627.47)
    Strict Risk of Losses:        0.00%
    ✅ APPROVED: Safe risk profile for capital investment.



In [ ]:
# --- OFFICIAL UNIT TESTS ---
print("🧪 Running Data Integrity Validation Tests...")

# Test 1: Ensure absolutely no negative quantities remain in the dataset
assert (data_final['Quantity'] <= 0).sum() == 0, "❌ TEST FAILED: Negative quantities found!"
print("✅ Test 1 Passed: All transaction quantities are strictly positive.")

# Test 2: Ensure absolutely no zero or negative unit prices remain
assert (data_final['UnitPrice'] <= 0).sum() == 0, "❌ TEST FAILED: Invalid unit prices found!"
print("✅ Test 2 Passed: All item unit prices are strictly positive.")

# Test 3: Ensure the target revenue matches exactly what was calculated
expected_rev = data_final['Quantity'] * data_final['UnitPrice']
assert np.allclose(data_final['Target_Revenue'], expected_rev), "❌ TEST FAILED: Target Revenue math mismatch!"
print("✅ Test 3 Passed: Target Revenue column calculations are 100% accurate.")

print("\n🚀 ALL CORES TESTS PASSED: Data structure is verified secure for production!")


🧪 Running Data Integrity Validation Tests...
✅ Test 1 Passed: All transaction quantities are strictly positive.
✅ Test 2 Passed: All item unit prices are strictly positive.
✅ Test 3 Passed: Target Revenue column calculations are 100% accurate.

🚀 ALL CORES TESTS PASSED: Data structure is verified secure for production!


# 📋 Project Documentation & Requirements Scope

### 1. Requirements & Business Scope
* **Objective:** Act as a Business Analytics Consultant to evaluate global market expansion. The goal is to determine the single most profitable country to launch an optimized targeted marketing campaign while maintaining a strict financial stability profile.
* **Operational Constraints:** A localized marketing budget of **\$5,000.00**. The campaign will target the top **200 highest-value transactions** predicted by a machine learning model.
* **Risk Tolerance:** Per corporate policy, any country with an estimated risk of financial loss exceeding **2.5%** must be automatically rejected.

### 2. Data Cleaning Pipeline & EDA
* **Source:** Real-world transactional logs from the official UCI Machine Learning Repository (Online Retail Dataset).
* **Exploratory Data Analysis (EDA):** A global `groupby` analysis identified that while the United Kingdom yielded the highest absolute transaction volume, continental European markets like the Netherlands and EIRE (Ireland) were heavily driven by massive high-value wholesale bulk orders.
* **Data Cleansing:** Successfully isolated and removed **10,624 canceled or returned orders** (negative quantities) and zero-priced error logs to prevent machine learning distortion.

### 3. Production Architecture & Implementation
* **Features Used:** `Quantity` and `UnitPrice`. 
* **Target Variable:** `Target_Revenue` (Quantity × UnitPrice).
* **Core Algorithms:** Independent `LinearRegression` frameworks trained for each localized country region utilizing a rigid 75:25 train-test split.
* **Risk Mitigation:** A 1,000-sample Bootstrap simulation loop was engineered to calculate exact expected net profits, 95% confidence intervals, and the strict mathematical risk of running at a loss.


WIREFRAME MARKDOWN
# ✏️ System Interface Wireframe Mockup Layout

+-----------------------------------------------------------------------+

|  💰 INTERNATIONAL MARKETING CAMPAIGN & RISK OPTIMIZATION DASHBOARD   |
+-----------------------------------------------------------------------+

|  [🔧 SIDEBAR PANEL CONTROL]       |  [📊 MAIN PRODUCTION PANEL]       |
|                                   |                                   |
|  * Slider: Investment Budget     |  * Phase 1: Machine Learning       |
|    [-- $5,000.00 -----------]     |    - Region 0 (UK) RMSE Valuation |
|                                   |    - Region 1 (NL) RMSE Valuation |
|  * Selector: Target Orders        |    - Region 2 (IE) RMSE Valuation |
|    [ 200 ]                        |                                   |
|                                   |  * Phase 2: Risk Bootstrapping    |
|  [ BUTTON: RUN SIMULATION ]       |    - Expected Net Profit Metrics  |
|                                   |    - Strict Risk of Loss % Box    |
|                                   |    - Probability Density Plot     |
+-----------------------------------------------------------------------+
